In [0]:
csv_path = "/Volumes/era5_project/raw/era5_files"

df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(csv_path)
)

print("Rows:", df_raw.count())
print("Columns:", len(df_raw.columns))

df_raw.printSchema()
display(df_raw.limit(10))

Rows: 507988
Columns: 7
root
 |-- Valid_time: string (nullable = true)
 |-- Pressure Level: integer (nullable = true)
 |-- Geopotential: double (nullable = true)
 |-- Relative Humidity: double (nullable = true)
 |-- Temperature: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)



Valid_time,Pressure Level,Geopotential,Relative Humidity,Temperature,latitude,longitude
1/01/2000 0:00,1000,742.0039,72.86274,294.68015,-36.75,174.75
1/01/2000 0:00,925,7303.4497,97.00138,288.02487,-36.75,174.75
1/01/2000 0:00,850,14285.859,83.205956,284.1772,-36.75,174.75
1/01/2000 0:00,700,29899.55,57.04135,273.3913,-36.75,174.75
1/01/2000 0:00,600,41893.16,21.778152,267.55777,-36.75,174.75
1/01/2000 0:00,500,55656.887,42.728226,257.56363,-36.75,174.75
1/01/2000 0:00,400,71803.875,38.95733,246.9512,-36.75,174.75
1/01/2000 0:00,300,91650.875,9.132197,234.95352,-36.75,174.75
1/01/2000 0:00,250,103841.375,4.7959237,230.73988,-36.75,174.75
1/01/2000 0:00,200,118408.69,5.3804083,224.43033,-36.75,174.75


In [0]:
from pyspark.sql import functions as F  # 1. Count total rows total_rows = df_ra ...

In [0]:
from pyspark.sql import functions as F

# Valid_time: string -> timestamp
df = df_raw.withColumn(
    "Valid_time",
    F.expr("try_to_timestamp(Valid_time, 'd/M/yyyy H:mm')")
)


df.printSchema()
display(df.limit(10))

bad_count = df.filter(F.col("Valid_time").isNull()).count() - df_raw.filter(F.col("Valid_time").isNull()).count()
print("Dòng lỗi định dạng:", bad_count)

root
 |-- Valid_time: timestamp (nullable = true)
 |-- Pressure Level: integer (nullable = true)
 |-- Geopotential: double (nullable = true)
 |-- Relative Humidity: double (nullable = true)
 |-- Temperature: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)



Valid_time,Pressure Level,Geopotential,Relative Humidity,Temperature,latitude,longitude
2000-01-01T00:00:00.000Z,1000,742.0039,72.86274,294.68015,-36.75,174.75
2000-01-01T00:00:00.000Z,925,7303.4497,97.00138,288.02487,-36.75,174.75
2000-01-01T00:00:00.000Z,850,14285.859,83.205956,284.1772,-36.75,174.75
2000-01-01T00:00:00.000Z,700,29899.55,57.04135,273.3913,-36.75,174.75
2000-01-01T00:00:00.000Z,600,41893.16,21.778152,267.55777,-36.75,174.75
2000-01-01T00:00:00.000Z,500,55656.887,42.728226,257.56363,-36.75,174.75
2000-01-01T00:00:00.000Z,400,71803.875,38.95733,246.9512,-36.75,174.75
2000-01-01T00:00:00.000Z,300,91650.875,9.132197,234.95352,-36.75,174.75
2000-01-01T00:00:00.000Z,250,103841.375,4.7959237,230.73988,-36.75,174.75
2000-01-01T00:00:00.000Z,200,118408.69,5.3804083,224.43033,-36.75,174.75


Dòng lỗi định dạng: 0


In [0]:
from pyspark.sql import functions as F

# 1.STANDADIZE
df = df.toDF(*[c.strip().lower().replace(" ", "_") for c in df.columns])
# -> valid_time, pressure_level, geopotential, relative_humidity, temperature, latitude, longitude


df = (
    df
    .withColumn("pressure_level", F.col("pressure_level").cast("int"))
    .withColumn("geopotential", F.col("geopotential").cast("double"))
    .withColumn("relative_humidity", F.col("relative_humidity").cast("double"))
    .withColumn("temperature", F.col("temperature").cast("double"))
    .withColumn("latitude", F.col("latitude").cast("double"))
    .withColumn("longitude", F.col("longitude").cast("double"))
)


df = (
    df
    .withColumn("temperature_c", F.col("temperature") - 273.15)
    .withColumn("geopotential_height_m", F.col("geopotential") / 9.80665)
)

# 4. CHECKING
df = df.dropDuplicates(["valid_time", "pressure_level", "latitude", "longitude"])

df.select(
    F.min("relative_humidity").alias("rh_min"), F.max("relative_humidity").alias("rh_max"),
    F.min("temperature_c").alias("t_min"), F.max("temperature_c").alias("t_max")
).show()

display(df.limit(10))

+----------+---------+------------------+------------------+
|    rh_min|   rh_max|             t_min|             t_max|
+----------+---------+------------------+------------------+
|-5.3504505|138.20152|-79.43211999999997|25.755460000000028|
+----------+---------+------------------+------------------+



valid_time,pressure_level,geopotential,relative_humidity,temperature,latitude,longitude,temperature_c,geopotential_height_m
2000-01-01T12:00:00.000Z,700,29865.586,36.79726,274.68002,-36.75,174.75,1.530020000000036,3045.4422254286633
2000-01-01T12:00:00.000Z,300,91958.31,14.784679,233.68431,-36.75,174.75,-39.46568999999997,9377.137962505036
2000-01-01T12:00:00.000Z,150,136592.44,4.848752,216.76363,-36.75,174.75,-56.38636999999997,13928.55256382149
2000-01-02T12:00:00.000Z,1000,691.7473,88.86223,289.4596,-36.75,174.75,16.309600000000046,70.53859370937069
2000-01-02T12:00:00.000Z,925,7145.2188,89.19455,284.51474,-36.75,174.75,11.36474000000004,728.6095455634697
2000-01-02T18:00:00.000Z,100,160713.31,6.1574097,209.5424,-36.75,174.75,-63.60759999999999,16388.19678483478
2000-01-03T18:00:00.000Z,150,135356.31,2.1437387,220.25763,-36.75,174.75,-52.89236999999997,13802.502383586649
2000-01-04T06:00:00.000Z,700,29176.879,37.93555,270.22485,-36.75,174.75,-2.9251499999999737,2975.2136560395243
2000-01-04T12:00:00.000Z,100,160463.75,6.189025,209.84145,-36.75,174.75,-63.30854999999997,16362.748747023705
2000-01-05T00:00:00.000Z,250,103380.125,37.95856,221.78758,-36.75,174.75,-51.362419999999986,10541.838956218486


In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("era5_project.silver.era5_clean")

In [0]:

df_check = spark.table("era5_project.silver.era5_clean")
df_check.printSchema()
print("Rows:", df_check.count())
display(df_check.limit(10))

root
 |-- valid_time: timestamp (nullable = true)
 |-- pressure_level: integer (nullable = true)
 |-- geopotential: double (nullable = true)
 |-- relative_humidity: double (nullable = true)
 |-- temperature: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- geopotential_height_m: double (nullable = true)

Rows: 507988


valid_time,pressure_level,geopotential,relative_humidity,temperature,latitude,longitude,temperature_c,geopotential_height_m
2000-01-01T12:00:00.000Z,700,29865.586,36.79726,274.68002,-36.75,174.75,1.530020000000036,3045.4422254286633
2000-01-01T12:00:00.000Z,300,91958.31,14.784679,233.68431,-36.75,174.75,-39.46568999999997,9377.137962505036
2000-01-01T12:00:00.000Z,150,136592.44,4.848752,216.76363,-36.75,174.75,-56.38636999999997,13928.55256382149
2000-01-02T12:00:00.000Z,1000,691.7473,88.86223,289.4596,-36.75,174.75,16.309600000000046,70.53859370937069
2000-01-02T12:00:00.000Z,925,7145.2188,89.19455,284.51474,-36.75,174.75,11.36474000000004,728.6095455634697
2000-01-02T18:00:00.000Z,100,160713.31,6.1574097,209.5424,-36.75,174.75,-63.60759999999999,16388.19678483478
2000-01-03T18:00:00.000Z,150,135356.31,2.1437387,220.25763,-36.75,174.75,-52.89236999999997,13802.502383586649
2000-01-04T06:00:00.000Z,700,29176.879,37.93555,270.22485,-36.75,174.75,-2.9251499999999737,2975.2136560395243
2000-01-04T12:00:00.000Z,100,160463.75,6.189025,209.84145,-36.75,174.75,-63.30854999999997,16362.748747023705
2000-01-05T00:00:00.000Z,250,103380.125,37.95856,221.78758,-36.75,174.75,-51.362419999999986,10541.838956218486


**==================PERFORANCE OPTIMIZATION==================
**



In [0]:
%sql
DESCRIBE DETAIL era5_project.silver.era5_clean;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,729a233a-835c-4b07-9ec9-1867cabacde7,era5_project.silver.era5_clean,"The table contains meteorological data related to various atmospheric parameters. It includes information such as temperature, pressure levels, geopotential height, and humidity, along with their corresponding geographical coordinates and timestamps. This data can be useful for climate analysis, weather prediction modeling, and studying atmospheric changes over time.",,2026-10-09T10:40:45.394Z,2026-10-30T11:31:04.000Z,List(),List(),1,15772092,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, io.unitycatalog.tableId -> 906100e3-09d6-4955-843f-443447c68092)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
display(dbutils.fs.ls("/Volumes/era5_project/raw/era5_files"))

path,name,size,modificationTime
dbfs:/Volumes/era5_project/raw/era5_files/ERA5.csv,ERA5.csv,32574585,1791537576000


In [0]:
%sql
SELECT COUNT(*) AS total_rows
FROM era5_project.silver.era5_clean;

total_rows
507988


In [0]:
from pyspark.sql import functions as F
import time
import statistics

source_table = "era5_project.silver.era5_clean"

# Change this to the exact catalog.schema.table name
# shown in Catalog Explorer for your summary table.
summary_table = "era5_project.silver.era5_profile_tempreture"


# A: calculate the aggregation from detailed records
def query_before():
    return (
        spark.table(source_table)
        .select(
            "pressure_level",
            "temperature_c",
            "geopotential_height_m"
        )
        .filter(
            F.col("pressure_level").isNotNull()
            & F.col("temperature_c").isNotNull()
            & F.col("geopotential_height_m").isNotNull()
        )
        .groupBy("pressure_level")
        .agg(
            F.count("temperature_c").alias("observation_count"),
            F.avg("temperature_c").alias("average_temperature_c"),
            F.avg("geopotential_height_m").alias(
                "average_geopotential_height_m"
            )
        )
        .withColumn(
            "average_geopotential_height_km",
            F.col("average_geopotential_height_m") / 1000
        )
        .orderBy("pressure_level")
    )


# B: read the pre-aggregated result
def query_after():
    return (
        spark.table(summary_table)
        .select(
            "pressure_level",
            "observation_count",
            "average_temperature_c",
            "average_geopotential_height_m",
            "average_geopotential_height_km"
        )
        .orderBy("pressure_level")
    )


def benchmark(query_function, repeats=5):
    times = []

    for i in range(repeats):
        start = time.perf_counter()
        rows = query_function().collect()
        elapsed = time.perf_counter() - start

        times.append(elapsed)
        print(f"Run {i + 1}: {elapsed:.3f} seconds")

    return statistics.median(times)


# Warm up both paths
query_before().collect()
query_after().collect()

print("BEFORE: aggregate detailed Silver data")
before_median = benchmark(query_before)

print("\nAFTER: read pre-aggregated Delta table")
after_median = benchmark(query_after)

print(f"\nBefore median: {before_median:.3f} seconds")
print(f"After median:  {after_median:.3f} seconds")

if before_median > 0:
    reduction_pct = (
        (before_median - after_median) / before_median
    ) * 100
    print(f"Runtime reduction: {reduction_pct:.2f}%")

BEFORE: aggregate detailed Silver data
Run 1: 0.903 seconds
Run 2: 0.573 seconds
Run 3: 0.616 seconds
Run 4: 0.556 seconds
Run 5: 0.532 seconds

AFTER: read pre-aggregated Delta table
Run 1: 0.473 seconds
Run 2: 0.437 seconds
Run 3: 0.454 seconds
Run 4: 0.431 seconds
Run 5: 0.505 seconds

Before median: 0.573 seconds
After median:  0.454 seconds
Runtime reduction: 20.90%
